# Rectangle Device Builder

Submit a mesh generation workflow to Argo, poll until done, then plot the result.

In [ ]:
import json
import uuid
import time
import httpx
import boto3
import tarfile
import io
import numpy as np
import plotly.graph_objects as go

from hera.workflows import Workflow, WorkflowsService, Parameter
from hera.workflows.models import WorkflowTemplateRef as WTR

## Connections

In [ ]:
gateway = "http://localhost:30080"

argo_svc = WorkflowsService(
    host=gateway,
    verify_ssl=False,
    namespace="tdgl",
)

minio = boto3.client(
    "s3",
    endpoint_url="http://localhost:30900",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
    region_name="us-east-1",
)

print(f"Gateway: {gateway}")
print(f"MinIO Console: {gateway}/minio-ui/")

## Device Parameters

In [ ]:
params = {
    "film_width": 10.0,
    "film_height": 6.0,
    "elec_width": 0.5,
    "elec_height": 4.1,
    "elec_y_offset": 0.0,
    "probe_points": [
        [-3.0, 0.0],
        [ 3.0, 0.0],
    ],
    "max_edge_length": 0.25,
    "smooth": 100,
}

## Submit Workflow

In [ ]:
run_id = str(uuid.uuid4())

wf = Workflow(
    generate_name="rect-device-",
    namespace="tdgl",
    workflow_template_ref=WTR(name="rectangle-device-builder"),
    arguments=[
        Parameter(name="run-id", value=run_id),
        Parameter(name="device-params-json", value=json.dumps(params)),
        Parameter(name="image", value="ghcr.io/fangrh/py-tdgl-runner:6262e5a"),
    ],
    workflows_service=argo_svc,
)

created = wf.create()
wf_name = created.metadata.name
print(f"Submitted: {wf_name}")
print(f"Run ID:    {run_id}")

## Poll Until Done

In [ ]:
hint_map = {
    "Submitted": "Scheduling...",
    "Pending": "Pulling image...",
    "Running": "Computing mesh...",
}

while True:
    url = f"{argo_svc.host}/api/v1/workflows/tdgl/{wf_name}"
    resp = httpx.get(url, verify=False, timeout=10)
    resp.raise_for_status()
    phase = (resp.json().get("status") or {}).get("phase", "Unknown")

    if phase == "Succeeded":
        print(f"{wf_name} succeeded.")
        break
    elif phase in {"Failed", "Error"}:
        raise RuntimeError(f"{wf_name} {phase}")
    else:
        hint = hint_map.get(phase, "Processing...")
        print(f"{wf_name} [{phase}] {hint}")
        time.sleep(3)

## Read Result from MinIO

In [ ]:
key = f"{run_id}/mesh_result.json"
resp = minio.get_object(Bucket="argo-artifacts", Key=key)
raw = resp["Body"].read()

mesh_result = None
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    for member in tar.getmembers():
        f = tar.extractfile(member)
        if f:
            mesh_result = json.loads(f.read())
            break

sites = np.array(mesh_result["sites"])
elements = np.array(mesh_result["elements"])
terminals = mesh_result.get("terminals", [])
probes = mesh_result.get("probe_indices", [])

print(f"Sites: {mesh_result['num_sites']}  Elements: {mesh_result['num_elements']}")
print(f"Film: {mesh_result.get('film_width')} x {mesh_result.get('film_height')}")
print(f"Probes: {probes}")

## Plot Mesh

In [ ]:
mx, my = [], []
for tri in elements:
    for j in range(3):
        p0, p1 = sites[tri[j]], sites[tri[(j + 1) % 3]]
        mx += [p0[0], p1[0], None]
        my += [p0[1], p1[1], None]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mx, y=my, mode="lines",
    line=dict(width=0.3, color="#94a3b8"),
    hoverinfo="skip", showlegend=False,
))

ec = {
    "source": ("#2563eb", "rgba(37,99,235,0.35)"),
    "drain": ("#dc2626", "rgba(220,38,38,0.35)"),
}
for t in terminals:
    idx = t["site_indices"]
    x0, x1 = sites[idx, 0].min(), sites[idx, 0].max()
    y0, y1 = sites[idx, 1].min(), sites[idx, 1].max()
    pad = 0.15
    lc, fc = ec.get(t["name"], ("#888", "rgba(136,136,136,0.35)"))
    fig.add_trace(go.Scatter(
        x=[x0 - pad, x1 + pad, x1 + pad, x0 - pad, x0 - pad],
        y=[y0 - pad, y0 - pad, y1 + pad, y1 + pad, y0 - pad],
        mode="lines", line=dict(width=1.5, color=lc), name=t["name"],
        fill="toself", fillcolor=fc,
    ))

fig.add_trace(go.Scatter(
    x=sites[probes, 0], y=sites[probes, 1],
    mode="markers+text",
    marker=dict(size=8, symbol="x", color="#16a34a", line_width=2),
    text=[f"P{i+1}" for i in range(len(probes))],
    textposition="top center", name="probes",
))

xmin, xmax = sites[:, 0].min(), sites[:, 0].max()
ymin, ymax = sites[:, 1].min(), sites[:, 1].max()
m = 0.3
fig.update_layout(
    title=f"Device ({len(sites)} sites, {len(elements)} elements)",
    xaxis=dict(range=[xmin - m, xmax + m], showline=True, linewidth=1,
               linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(scaleanchor="x", scaleratio=1, range=[ymin - m, ymax + m],
               showline=True, linewidth=1, linecolor="black", mirror=True, ticks="outside"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    margin=dict(l=40, r=10, t=35, b=50),
    height=280, width=700, plot_bgcolor="white",
)
fig

## MinIO Artifact Management

In [ ]:
# Delete the current run's artifact
minio.delete_object(Bucket="argo-artifacts", Key=key)
print(f"Deleted {key}")

In [ ]:
# List all artifacts in the bucket
resp = minio.list_objects_v2(Bucket="argo-artifacts")
items = resp.get("Contents", [])

if not items:
    print("No artifacts in bucket.")
else:
    print(f"Found {len(items)} artifact(s):\n")
    for obj in items:
        print(f"  {obj['Key']}  ({obj['Size']/1024:.1f} KB)")

In [ ]:
# Delete a specific artifact — change the key below
delete_key = "<run-id>/mesh_result.json"  # e.g. "abc123/mesh_result.json"

minio.delete_object(Bucket="argo-artifacts", Key=delete_key)
print(f"Deleted {delete_key}")